# Step 00 - Environment Setup

Pulls code from the public GitHub repo and verifies imports before running scoring/analysis notebooks.

1. Set `REPO_URL` and clone/pull the repo
2. Install requirements and add `src/` to `sys.path`
3. Validate imports
4. Use monitoring cells while long scoring jobs are running

In [9]:
# Cell 0 — always run first
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/shreetishresthanp/latent_verbal_gap.git"
WORKDIR = Path("/content")
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
REPO = WORKDIR / REPO_NAME

if "<owner>" in REPO_URL or "<repo>" in REPO_URL:
    raise ValueError("Please set REPO_URL to your public GitHub repo before running.")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated existing repo: {REPO}")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    print(f"Cloned repo: {REPO}")

Updated existing repo: /content/latent_verbal_gap


In [10]:
import sys

REQ_FILE = REPO / "requirements.txt"
SRC_DIR = REPO / "src"

# Install project dependencies
if REQ_FILE.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ_FILE)],
        check=True,
    )
else:
    print(f"Skipping install: missing {REQ_FILE}")

# Add src/ once per runtime
src_path = str(SRC_DIR)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Using src at: {SRC_DIR}")
print("Setup complete")

Skipping install: missing /content/latent_verbal_gap/requirements.txt
Using src at: /content/latent_verbal_gap/src
Setup complete


In [14]:
# Optional: only needed if your runtime does not already include python-dotenv
%pip install -q python-dotenv

In [ ]:
%ls

Latent_Verbal_Gap_Proposal.pdf  README.md


In [ ]:
# Optional cell created to check intermediate progress when running scoring
import pandas as pd
import config

TOTAL_EXPECTED_ROWS = 5_994
SCORES_TMP_FILE = config.CACHE_DIR / "lva_scores_retry_tmp.parquet"

if SCORES_TMP_FILE.exists():
    df = pd.read_parquet(SCORES_TMP_FILE)
    error_count = (df["lva_score"] < 0).sum()

    print(f"Done: {len(df):,} / {TOTAL_EXPECTED_ROWS:,}")
    print(f"Errors: {error_count}")
    print("\nSample of recent scores:")
    print(
        df.tail(5)[
            ["sequence_id", "step_idx", "feature_label", "lva_score", "confidence"]
        ].to_string()
    )
else:
    print(f"Checkpoint not found: {SCORES_TMP_FILE.name}")

Done: 200 / 5994:,
Errors: 0

Sample of recent scores:
     sequence_id  step_idx                                            feature_label  lva_score confidence
195        35000       253  Mathematical reasoning and step-by-step problem solving        0.4       high
196        36000        10                   Mathematical problem-solving reasoning        0.4       high
197        36000        28        Mathematical problem-solving with formal notation        0.4       high
198        36000        32                Mathematical problem-solving and counting        0.4       high
199        36000        68  Mathematical reasoning and step-by-step problem solving        0.4       high
